In [ ]:
# %% [Cell 1] Imports & config
import hashlib
import json
import re
import uuid
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.cloud import bigquery
from pypdf import PdfReader

# Run `!pip install google-cloud-aiplatform` above if not already available.
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"
GEMINI_MODEL = "gemini-2.5-pro"
GEMINI_LOCATION = "us-central1"  # most common region for Gemini on Vertex AI — confirm
                                   # this is actually enabled for your project; your BQ
                                   # dataset is in us-west1, which doesn't tell us anything
                                   # about Vertex AI region availability/policy
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_uploadManifest_weekly"
BRONZE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_pdfPages_weekly"
SILVER_CI_HEADLINES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_ciHeadlines_weekly"
SILVER_RETAIL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_nationalRetailReadout_weekly"

bq = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=GEMINI_LOCATION)


# %% [Cell 1b] Create tables if they don't exist yet — idempotent, safe to re-run every time
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{MANIFEST_TABLE}` (
        report_date DATE,
        run_id STRING,
        file_name STRING,
        file_hash STRING,
        status STRING,         -- 'active' | 'superseded' | 'cancelled'
        uploaded_at TIMESTAMP,
        uploaded_by STRING
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{BRONZE_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_pdf STRING,
        page_number INT64,
        page_type STRING,
        text_extraction_raw STRING,
        llm_extraction_raw STRING,
        page_image_gcs_uri STRING,
        prompt_version STRING,
        extracted_at TIMESTAMP
    )
""").result()

print("Manifest and bronze tables ready.")


# %% [Cell 1c] One-time cleanup — NOT run automatically. Call reset_tables() yourself,
# from a cell below, only if you hit a "streaming buffer" error on delete (rows written
# before this notebook switched to load jobs, or written moments ago, can get stuck
# there for a while). This drops all data in both tables, so treat it as a reset
# button for test data, not something to run against a table you actually care about.
def reset_tables():
    bq.query(f"DROP TABLE IF EXISTS `{BRONZE_TABLE}`").result()
    bq.query(f"DROP TABLE IF EXISTS `{MANIFEST_TABLE}`").result()
    print("Dropped both tables. Re-run Cell 1b to recreate them, then re-upload your files.")

# reset_tables()  # <- uncomment and run manually when needed


# %% [Cell 2] Helpers: parse date, hash file, check manifest, write manifest row

def parse_date_from_filename(filename: str):
    """MCI's convention: '..._MMDDYY.pdf' e.g. Competitive_Offer_Report_050826.pdf"""
    m = re.search(r"_(\d{2})(\d{2})(\d{2})\.pdf$", filename, re.IGNORECASE)
    if not m:
        return None
    mm, dd, yy = m.groups()
    return datetime.strptime(f"20{yy}-{mm}-{dd}", "%Y-%m-%d").date()


def verify_date_from_content(pdf_bytes: bytes, expected_date):
    """Cross-check the filename-derived date against text actually inside the PDF
    (CI Headlines page reliably contains 'COMPETITIVE INTELLIGENCE HEADLINES <Month D, YYYY>').
    Cheap: only reads a couple of pages, not the whole deck."""
    from io import BytesIO
    reader = PdfReader(BytesIO(pdf_bytes))
    for page in reader.pages[:6]:
        text = page.extract_text() or ""
        m = re.search(r"([A-Z][a-z]+ \d{1,2}, 20\d{2})", text)
        if m:
            found = datetime.strptime(m.group(1), "%B %d, %Y").date()
            return found == expected_date, found
    return None, None  # couldn't verify — not necessarily a problem, just flag as unverified


def file_hash(pdf_bytes: bytes) -> str:
    return hashlib.sha256(pdf_bytes).hexdigest()


def get_active_manifest_row(report_date):
    query = f"""
        SELECT run_id, file_name, file_hash, uploaded_at
        FROM `{MANIFEST_TABLE}`
        WHERE report_date = @report_date AND status = 'active'
        ORDER BY uploaded_at DESC
        LIMIT 1
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    )
    rows = list(job.result())
    return rows[0] if rows else None


def write_row(table_id, row: dict):
    """Load-job insert instead of streaming (insert_rows_json) — load jobs write
    directly to the table with no streaming buffer, so a DELETE run shortly after
    (e.g. while testing) won't hit BigQuery's 'can't DML rows in streaming buffer'
    error."""
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json([row], table_id, job_config=job_config)
    job.result()  # raises on failure


def write_rows(table_id, rows: list):
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json(rows, table_id, job_config=job_config)
    job.result()


def write_manifest_row(report_date, run_id, file_name, hash_, status):
    row = {
        "report_date": report_date.isoformat(),
        "run_id": run_id,
        "file_name": file_name,
        "file_hash": hash_,
        "status": status,
        "uploaded_at": datetime.utcnow().isoformat(),
        "uploaded_by": "notebook1",
    }
    write_row(MANIFEST_TABLE, row)


def delete_existing_run(report_date) -> bool:
    """Literal delete-then-replace, as requested — removes both the old bronze rows
    and the old manifest row for this report_date before the new run writes its own.
    Returns False if it couldn't complete, so the caller can abort rather than write
    new data on top of old data that failed to clear."""
    for table_id in (BRONZE_TABLE, MANIFEST_TABLE):
        try:
            bq.query(
                f"DELETE FROM `{table_id}` WHERE report_date = @report_date",
                job_config=bigquery.QueryJobConfig(
                    query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
                ),
            ).result()
        except Exception as e:
            if "streaming buffer" in str(e):
                print(f"Can't delete from {table_id} yet — those rows are still in BigQuery's "
                      f"streaming buffer (usually clears within a couple hours on its own). "
                      f"If this is test data and you don't want to wait, run reset_tables() "
                      f"(defined in Cell 1c) and re-upload.")
                return False
            raise
    return True


# %% [Cell 3] Stage 1: PDF -> per-page text extraction -> page-type classification -> bronze write
# Run `!pip install pymupdf` above if fitz isn't already available in this environment.
# NOTE: Gemini extraction and the silver-layer writes are deliberately NOT in this
# version — that's the next increment, once bronze output below looks right.
import fitz  # PyMuPDF

# First pass, built from titles observed in your two sample files. Anything that
# doesn't match falls through to 'unclassified' rather than being guessed at —
# check the printed output for those and add a pattern instead of assuming.
PAGE_TYPE_PATTERNS = [
    (r"Table of Contents", "toc"),
    (r"NOTICE OF CONFIDENTIALITY", "confidentiality_notice"),
    (r"COMPETITIVE INTELLIGENCE HEADLINES", "ci_headlines"),
    (r"CI Headlines", "section_divider"),  # short divider title, distinct from the phrase above
    (r"Major Offer Changes Effective", "major_offer_changes"),
    (r"Post-Paid Apple Smartphone Offers", "postpaid_device_offers_apple"),
    (r"Post-Paid Samsung and Google Smartphone Offers", "postpaid_device_offers_samsung_google"),
    (r"Post-Paid Motorola, Affordable Phones and BYOD", "postpaid_device_offers_motorola_affordable"),
    (r"Post-Paid BTS \(Watches\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS \(Tablet\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS & Other Offers", "postpaid_bts_offers"),
    (r"Post-Paid Service Offers", "postpaid_service_offers"),
    (r"Postpaid Cable MVNO Handset Offers", "cable_mvno_offers"),
    (r"Cable MVNO Service Pricing", "cable_mvno_offers"),
    (r"Business\s*[-–]\s*Flagship Device Offers", "business_device_offers"),
    (r"Key Highlights\s*[-–]\s*Week Ending", "national_retail_readout"),  # more reliable than the
    (r"Promotions Marketplace", "national_retail_readout"),               # Circana wordmark, which
                                                                            # may render as an image
    (r"Prepaid Brands\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Flagship Brands/Prepaid\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Walmart\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"METRO BY T-MOBILE", "prepaid_price_grid_detail"),
    (r"Competitive Offers\s*&\s*Promotional\s*Updates", "section_divider"),
    (r"Competitive Offer Updates", "section_divider"),  # singular, no "& Promotional" — different divider
    (r"Competitive Offer Report", "title_slide"),
    (r"ARE YOU WITH US", "closing_slide"),
]


def classify_page(text: str):
    normalized = re.sub(r"\s+", " ", text)  # collapse newlines/multi-space so line-wrap
                                              # differences between slide instances don't matter
    for pattern, page_type in PAGE_TYPE_PATTERNS:
        if re.search(pattern, normalized, re.IGNORECASE):
            return page_type, pattern
    return "unclassified", None


# Pages with no offer content — the Gemini extraction stage (next increment) should
# skip these entirely rather than pay for a model call with nothing to extract.
# These are also exactly the pages with the duplicated-text rendering artifact
# (shadow-effect titles over photo backgrounds), so skipping them sidesteps that
# problem too rather than needing to solve it.
NON_CONTENT_PAGE_TYPES = {
    "title_slide", "confidentiality_notice", "toc", "section_divider", "closing_slide",
}


def reconstruct_layout_text(page, y_tolerance=3):
    """Cluster words into horizontal rows by y-position, sort left-to-right within
    each row — this replaces pypdf's layout mode, which produced empty table bodies
    on your files. y_tolerance is a starting guess: if printed rows below look merged
    or split wrong on a given page, that's the value to adjust."""
    words = page.get_text("words")  # (x0, y0, x1, y1, word, block, line, word_no)
    if not words:
        return ""
    words = sorted(words, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    lines, current_bucket, current_line = [], None, []
    for w in words:
        bucket = round(w[1] / y_tolerance)
        if current_bucket is None or bucket == current_bucket:
            current_line.append(w)
            current_bucket = bucket
        else:
            lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
            current_line, current_bucket = [w], bucket
    if current_line:
        lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
    return "\n".join(lines)


KNOWN_COLUMN_HEADERS = {
    "t-mobile", "verizon", "at&t", "att", "xfinity", "spectrum", "comcast",
    "optimum", "metro", "boost", "cricket", "smartphones",
}


def find_header_columns(page, y_tolerance=3, min_words_in_row=3, min_gap=15):
    """Look for the column-header row in the top third of the page. Two signals,
    combined: (1) skip the very first row-cluster, since on every page checked so
    far that's the page title, not the header row underneath it; (2) among what's
    left, prefer rows containing actual known carrier names over rows that just
    happen to have the most spaced-out words — word count alone picked the title
    over the real header on page 10, which is what this is fixing."""
    words = page.get_text("words")
    if not words:
        return None
    page_height = page.rect.height
    candidates = [w for w in words if w[1] < page_height * 0.35]
    candidates = sorted(candidates, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    rows = {}
    for w in candidates:
        bucket = round(w[1] / y_tolerance)
        rows.setdefault(bucket, []).append(w)

    sorted_buckets = sorted(rows.keys())
    if len(sorted_buckets) > 1:
        rows = {b: rows[b] for b in sorted_buckets[1:]}  # drop the title row

    scored = []
    for row_words in rows.values():
        row_words = sorted(row_words, key=lambda w: w[0])
        if len(row_words) < min_words_in_row:
            continue
        gaps = [row_words[i + 1][0] - row_words[i][2] for i in range(len(row_words) - 1)]
        if not gaps or min(gaps) <= min_gap:
            continue
        vocab_hits = sum(1 for w in row_words if w[4].lower().strip(":") in KNOWN_COLUMN_HEADERS)
        scored.append((vocab_hits, len(row_words), row_words))

    if not scored:
        return None
    scored.sort(key=lambda t: (t[0], t[1]), reverse=True)  # vocab matches win, word count as tiebreak
    return [w[0] for w in scored[0][2]]


def reconstruct_columns(page, y_tolerance=3):
    """Column-aware alternative to reconstruct_layout_text, for grid-table pages.
    Detects column boundaries from a header row, buckets every word into its
    column by x-position FIRST, then does row-clustering within each column
    separately. This avoids the cross-column bleeding seen when clustering by
    y-position across the full page width — different carriers wrapping to
    different numbers of lines for the same conceptual row was zippering
    unrelated columns' text together."""
    boundaries = find_header_columns(page, y_tolerance=y_tolerance)
    if not boundaries:
        return reconstruct_layout_text(page, y_tolerance=y_tolerance)  # fallback

    words = page.get_text("words")
    columns = {i: [] for i in range(len(boundaries))}
    for w in words:
        col_idx = 0
        for i, b in enumerate(boundaries):
            if w[0] >= b:
                col_idx = i
        columns[col_idx].append(w)

    blocks = []
    for i in range(len(boundaries)):
        col_words = sorted(columns[i], key=lambda w: (round(w[1] / y_tolerance), w[0]))
        lines, bucket, line = [], None, []
        for w in col_words:
            b = round(w[1] / y_tolerance)
            if bucket is None or b == bucket:
                line.append(w)
                bucket = b
            else:
                lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
                line, bucket = [w], b
        if line:
            lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
        blocks.append(f"[COLUMN {i + 1}]\n" + "\n".join(lines))
    return "\n\n".join(blocks)


def run_extraction_pipeline(run_id: str, report_date, pdf_bytes: bytes, file_name: str):
    print(f"[{run_id}] Opening {file_name} ...")
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    print(f"[{run_id}] {len(doc)} pages found.\n")

    rows = []
    total_pages = len(doc)
    for i, page in enumerate(doc):
        page_num = i + 1
        layout_text = reconstruct_layout_text(page)
        page_type, matched_pattern = classify_page(layout_text)

        if page_type == "unclassified" and page_num == 1:
            # Same reasoning as the closing-slide fallback below: title slide text is
            # duplicated by what looks like a shadow-effect rendering (two overlapping
            # text boxes), which breaks literal-phrase matching unpredictably depending
            # on block order. Position is the reliable signal here, not the text.
            page_type, matched_pattern = "title_slide", "(fallback: first page of deck)"

        if page_type == "unclassified" and page_num == total_pages:
            # No text pattern is reliable here — the closing slide is a full-bleed
            # brand graphic with no real title text. Position relative to the deck's
            # own length (not an absolute page number) is safe to use as a fallback.
            page_type, matched_pattern = "closing_slide", "(fallback: last page of deck)"

        print(f"--- Page {page_num} ---")
        print(f"  matched pattern : {matched_pattern!r}")
        print(f"  page_type       : {page_type}")
        print(f"  text preview    : {layout_text[:120]!r}")
        print()

        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_pdf": file_name,
            "page_number": page_num,
            "page_type": page_type,
            "text_extraction_raw": layout_text,
            "llm_extraction_raw": None,   # next increment: Gemini call
            "page_image_gcs_uri": None,   # next increment
            "prompt_version": None,       # next increment
            "extracted_at": datetime.utcnow().isoformat(),
        })

    unclassified = [r["page_number"] for r in rows if r["page_type"] == "unclassified"]
    if unclassified:
        print(f"[{run_id}] WARNING: {len(unclassified)} page(s) matched no known pattern: {unclassified}")
        print("These are still written to bronze as 'unclassified' — inspect them and add a pattern if they're real content pages, not just dividers/photos.\n")

    errors = None
    try:
        write_rows(BRONZE_TABLE, rows)
    except Exception as e:
        errors = e
    if errors:
        print(f"[{run_id}] BigQuery write failed: {errors}")
    else:
        print(f"[{run_id}] Wrote {len(rows)} rows to bronze.")


# %% [Cell 4] Upload widget + explicit Upload button + dedup check + decision UI

upload_widget = widgets.FileUpload(accept=".pdf", multiple=False, description="Choose file")
upload_button = widgets.Button(description="Upload", button_style="primary", icon="upload")
output = widgets.Output()
display(widgets.HBox([upload_widget, upload_button]), output)


def proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode):
    with output:
        if mode == "cancel":
            print("Cancelled — nothing written.")
            return
        if mode == "replace":
            print(f"Deleting existing data for {report_date} ...")
            if not delete_existing_run(report_date):
                print("Aborting — old data wasn't fully cleared, so nothing new was written (avoids duplicates).")
                return
        write_manifest_row(report_date, run_id, file_name, hash_, status="active")
        run_extraction_pipeline(run_id, report_date, pdf_bytes, file_name)


_last_pdf_bytes = None
_last_file_name = None


def handle_upload(b):
    global _last_pdf_bytes, _last_file_name
    output.clear_output()
    if not upload_widget.value:
        with output:
            print("No file selected yet — choose a PDF first, then click Upload.")
        return

    uploaded = list(upload_widget.value.values())[0] if hasattr(upload_widget.value, "values") else upload_widget.value[0]
    file_name = uploaded["metadata"]["name"] if "metadata" in uploaded else uploaded.name
    pdf_bytes = uploaded["content"] if "content" in uploaded else uploaded.content
    _last_pdf_bytes, _last_file_name = pdf_bytes, file_name

    with output:
        report_date = parse_date_from_filename(file_name)
        if report_date is None:
            print(f"Couldn't parse a report date from filename '{file_name}' — check naming convention.")
            return

        match, found_date = verify_date_from_content(pdf_bytes, report_date)
        if match is False:
            print(f"Warning: filename implies {report_date}, but content suggests {found_date}. Proceeding with content date.")
            report_date = found_date

        hash_ = file_hash(pdf_bytes)
        existing = get_active_manifest_row(report_date)
        run_id = str(uuid.uuid4())

        if existing is None:
            print(f"No existing data for {report_date}. Proceeding.")
            proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode="new")
            return

        print(f"Data for {report_date} already exists (run {existing['run_id']}, file '{existing['file_name']}').")
        print("Choose how to proceed:")

        replace_btn = widgets.Button(description="Replace", button_style="danger")
        append_btn = widgets.Button(description="Append", button_style="warning")
        cancel_btn = widgets.Button(description="Cancel", button_style="")
        display(widgets.HBox([replace_btn, append_btn, cancel_btn]))

        replace_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "replace"))
        # NOTE on append: this leaves two 'active' manifest rows for the same report_date
        # (nothing gets deleted). Decide before relying on this: should gold then show
        # both runs, or should "append" in your workflow actually behave like replace
        # (e.g. if it's really just MCI sending a corrected re-issue of the same week)?
        append_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "append"))
        cancel_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "cancel"))


upload_button.on_click(handle_upload)


# %% [Cell 5] Diagnostic: full text reconstruction for one page, not just the 120-char
# preview. method="rows" is what's currently in the pipeline (Cell 3) — the one that
# just showed cross-column bleeding on page 10. method="columns" is the new
# column-aware alternative. Compare both on the same page before deciding which one
# actually goes into run_extraction_pipeline.
def inspect_page(page_number: int, method="columns", y_tolerance=3, pdf_bytes=None):
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first, or pass pdf_bytes explicitly.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc[page_number - 1]
    text = reconstruct_columns(page, y_tolerance) if method == "columns" else reconstruct_layout_text(page, y_tolerance)
    print(text)

# inspect_page(10, method="columns")


# %% [Cell 6] Extraction-strategy tiers, and a full-deck dump for reviewing every
# page_type in one pass. GRID types get column-aware reconstruction (matches what
# they'll get: image + text in the Gemini stage). Everything else content-bearing
# gets plain row reconstruction (matches what they'll get: text-only). Non-content
# types are skipped, same as they will be in the real extraction stage.
GRID_PAGE_TYPES = {
    "postpaid_device_offers_apple",
    "postpaid_device_offers_samsung_google",
    "postpaid_device_offers_motorola_affordable",
    "postpaid_bts_offers",
    "postpaid_service_offers",
    "cable_mvno_offers",
    "business_device_offers",
    "prepaid_brand_offers",
    "prepaid_price_grid_detail",
    "major_offer_changes",
}


def dump_all_pages(pdf_bytes=None, y_tolerance=3):
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    total_pages = len(doc)

    for i, page in enumerate(doc):
        page_num = i + 1
        row_text = reconstruct_layout_text(page, y_tolerance)
        page_type, _ = classify_page(row_text)
        if page_type == "unclassified" and page_num == 1:
            page_type = "title_slide"
        if page_type == "unclassified" and page_num == total_pages:
            page_type = "closing_slide"

        strategy = (
            "skip" if page_type in NON_CONTENT_PAGE_TYPES
            else "text+image" if page_type in GRID_PAGE_TYPES
            else "text-only"
        )

        print(f"\n{'=' * 80}\nPAGE {page_num}  |  page_type: {page_type}  |  strategy: {strategy}\n{'=' * 80}")

        if strategy == "skip":
            print("(non-content page, no extraction planned)")
            continue

        print(reconstruct_columns(page, y_tolerance) if strategy == "text+image" else row_text)

# dump_all_pages()


# %% [Cell 7] Stage 2, text-only tier: Gemini extraction for ci_headlines and
# national_retail_readout. Reads from bronze (not the PDF directly), so this can
# be re-run against an existing run_id without re-uploading or re-parsing.

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_CI_HEADLINES_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        carrier STRING,
        section STRING,
        headline_title STRING,
        headline_detail STRING,
        effective_date_range STRING,
        has_changes BOOL,
        values_agree BOOL,
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_RETAIL_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        week_ending STRING,
        retailer STRING,
        device STRING,
        discount_amount STRING,
        final_price STRING,
        carrier_service STRING,
        note STRING,
        values_agree BOOL,
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

print("Text-only silver tables ready.")


CI_HEADLINES_SCHEMA = {
    "type": "object",
    "properties": {
        "headlines": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"},
                    "section": {"type": "string"},
                    "headline_title": {"type": "string"},
                    "headline_detail": {"type": "string"},
                    "effective_date_range": {"type": "string"},
                    "has_changes": {"type": "boolean"},
                },
                "required": ["carrier", "headline_title", "has_changes"],
            },
        }
    },
    "required": ["headlines"],
}

CI_HEADLINES_PROMPT = """You are extracting structured data from a competitive \
intelligence report page. The page contains a numbered list of carriers, each \
followed by either specific bulleted headline items describing changes, or a \
simple "No changes" note.

For EACH numbered carrier entry:
- If it says "No changes" (or equivalent), emit exactly one record with \
has_changes=false, headline_title="No changes", headline_detail="".
- Otherwise, emit one record per bullet point, with has_changes=true.

Extract "section" from the page's subheading (e.g. "Postpaid & Cable MVNO Key \
Highlights"). Extract any date or date range in parentheses into \
effective_date_range if present, otherwise leave it blank. Do not paraphrase or \
summarize headline_detail — keep it close to the source wording, since this \
feeds a business report.

PAGE TEXT:
{page_text}"""


RETAIL_READOUT_SCHEMA = {
    "type": "object",
    "properties": {
        "week_ending": {"type": "string"},
        "retail_offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "retailer": {"type": "string"},
                    "device": {"type": "string"},
                    "discount_amount": {"type": "string"},
                    "final_price": {"type": "string"},
                    "carrier_service": {"type": "string"},
                    "note": {"type": "string"},
                },
                "required": ["retailer", "device"],
            },
        },
    },
    "required": ["retail_offers"],
}

RETAIL_READOUT_PROMPT = """You are extracting structured retail promotion data \
from a weekly competitive intelligence report page. The page is prose-style \
bullets describing which retailers are running which device discounts.

Extract "week_ending" from the page header. For each distinct retailer+device \
offer mentioned, extract one record with: retailer, device (model name only), \
discount_amount (as a dollar string, e.g. "$350"), final_price if mentioned, \
carrier_service if the offer is tied to a specific carrier/MVNO (otherwise \
"unlocked"), and a short note for any other relevant detail (e.g. comparison to \
last week). If a bullet mentions no specific offer (e.g. "no new promotions"), \
skip it — don't emit an empty record.

PAGE TEXT:
{page_text}"""


def call_gemini_json(prompt: str, schema: dict):
    model = GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=0,  # minimize drift on what should be a factual extraction
        ),
    )
    return json.loads(response.text)


def values_present_in_text(record: dict, keys: list, source_text: str) -> bool:
    """Cheap hallucination guard — checks that key extracted values literally
    appear in the bronze text for this page. Doesn't require our own extraction
    to be structurally perfect, just present."""
    for key in keys:
        val = str(record.get(key, "")).strip()
        if val and val not in source_text:
            return False
    return True


def fetch_bronze_pages(run_id, page_types):
    query = f"""
        SELECT page_number, page_type, text_extraction_raw
        FROM `{BRONZE_TABLE}`
        WHERE run_id = @run_id AND page_type IN UNNEST(@page_types)
        ORDER BY page_number
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                bigquery.ArrayQueryParameter("page_types", "STRING", page_types),
            ]
        ),
    )
    return list(job.result())


def extract_ci_headlines(page_number, page_text, report_date, run_id):
    result = call_gemini_json(CI_HEADLINES_PROMPT.format(page_text=page_text), CI_HEADLINES_SCHEMA)
    rows = []
    for h in result.get("headlines", []):
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "carrier": h.get("carrier"),
            "section": h.get("section"),
            "headline_title": h.get("headline_title"),
            "headline_detail": h.get("headline_detail"),
            "effective_date_range": h.get("effective_date_range"),
            "has_changes": h.get("has_changes"),
            "values_agree": values_present_in_text(h, ["carrier"], page_text),
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
    return rows


def extract_national_retail(page_number, page_text, report_date, run_id):
    result = call_gemini_json(RETAIL_READOUT_PROMPT.format(page_text=page_text), RETAIL_READOUT_SCHEMA)
    week_ending = result.get("week_ending")
    rows = []
    for o in result.get("retail_offers", []):
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "week_ending": week_ending,
            "retailer": o.get("retailer"),
            "device": o.get("device"),
            "discount_amount": o.get("discount_amount"),
            "final_price": o.get("final_price"),
            "carrier_service": o.get("carrier_service"),
            "note": o.get("note"),
            "values_agree": values_present_in_text(o, ["retailer", "device", "discount_amount"], page_text),
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
    return rows


def run_text_only_extraction(run_id, report_date):
    print(f"[{run_id}] Text-only extraction stage starting...")
    ci_rows, retail_rows = [], []

    for page in fetch_bronze_pages(run_id, ["ci_headlines"]):
        print(f"\n--- Page {page['page_number']} (ci_headlines) ---")
        try:
            rows = extract_ci_headlines(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print(f"  extracted {len(rows)} headline records")
            for r in rows:
                flag = "OK   " if r["values_agree"] else "CHECK"
                print(f"    [{flag}] {r['carrier']}: {r['headline_title']}")
            ci_rows.extend(rows)
        except Exception as e:
            print(f"  FAILED: {e}")

    for page in fetch_bronze_pages(run_id, ["national_retail_readout"]):
        print(f"\n--- Page {page['page_number']} (national_retail_readout) ---")
        try:
            rows = extract_national_retail(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print(f"  extracted {len(rows)} retail offer records")
            for r in rows:
                flag = "OK   " if r["values_agree"] else "CHECK"
                print(f"    [{flag}] {r['retailer']}: {r['device']} ({r['discount_amount']})")
            retail_rows.extend(rows)
        except Exception as e:
            print(f"  FAILED: {e}")

    if ci_rows:
        write_rows(SILVER_CI_HEADLINES_TABLE, ci_rows)
        print(f"\n[{run_id}] Wrote {len(ci_rows)} rows to silver ci_headlines.")
    if retail_rows:
        write_rows(SILVER_RETAIL_TABLE, retail_rows)
        print(f"[{run_id}] Wrote {len(retail_rows)} rows to silver national_retail_readout.")

# run_text_only_extraction(run_id="...", report_date=date(2026, 5, 29))


# %% [Cell 8] Diagnostic: what's actually in bronze right now, and does the
# run_id you're using match it. Run this before re-trying run_text_only_extraction.
def show_bronze_state():
    query = f"""
        SELECT run_id, report_date, page_type, COUNT(*) AS page_count
        FROM `{BRONZE_TABLE}`
        GROUP BY run_id, report_date, page_type
        ORDER BY report_date DESC, run_id, page_type
    """
    for row in bq.query(query).result():
        print(f"{row['report_date']}  |  run_id {row['run_id']}  |  {row['page_type']:35s}  |  {row['page_count']} page(s)")

show_bronze_state()